# 📝 과제 01: 회의록을 실행계획으로 정리하는 AI 업무 비서를 만듭니다

회의록으로 계획을 작성·검토·수정하고, 회의별 후속 질문에 답합니다.

<img src="images/assignment01_overview.png" width="1050" alt="회의록 AI 업무 비서의 입력, 작업 노드, 분기와 저장 흐름">

**진행:** 작성 → 검토 → 필요시 수정·재검토 → 결과 보관 → 회의별 후속 상담.

| 문항 | 작업 | 작성 방식 |
|---|---|---|
| 1 | State·초기 입력 | 전체 작성 |
| 2~4 | 작성·검토·수정 노드 | 골격 완성 |
| 5 | 종료 조건 | 전체 작성 |
| 6~9 | 연결·실행·독립 작업·이력 | 골격 완성 |
| 10 | 후속 상담 State | 전체 작성 |
| 11~12 | 상담 그래프·대화 분리 | 골격 완성 |

문항을 순서대로 완성합니다. 긴 입력·프롬프트와 출력 확인은 제공하며, 함수 정의만으로 모델이 실행되지는 않습니다. 노드는 바뀌는 필드만 반환합니다. 자료는 모두 학습용 가상 회의록입니다.

**모델 호출:** A는 검토 상한 3회로 최대 6회, B는 상한 1회로 2회, 수정 경로 확인은 2회, 후속 상담은 3회입니다. 지정된 실행은 최대 13회이며 실행 셀을 다시 돌리면 추가 호출됩니다. 자가평가는 고정 응답으로 코드를 검사하며 GPT를 추가 호출하지 않습니다. 모델의 검토 통과가 사실 일치를 보장하지 않으므로 원문과 직접 대조합니다.

## 준비: 두 회의의 자료를 읽습니다

**공통 작성 규칙:** 노드는 입력 State와 그 안의 목록을 직접 변경하지 않고, 갱신할 필드만 반환합니다. 호출 입력은 위치 인자와 키워드 인자 어느 쪽으로 전달해도 됩니다.

**자가평가:** 각 문항을 작성한 뒤 `assert` 셀을 실행하세요. 노드 정의는 제공된 고정 응답으로 입출력을 검사하고, 실행 문항은 실제 실행 결과를 검사합니다. 자가평가 셀 자체는 GPT를 추가 호출하지 않습니다. 통과 후에도 생성된 글의 사실과 표현은 원자료와 직접 대조하세요.

.env와 모델을 준비하는 제공 셀입니다. 그대로 실행하세요. 이 단계에서는 모델을 호출하지 않습니다.

In [ ]:
# 교안과 같은 환경 설정과 모델을 사용합니다. 이 셀은 모델을 호출하지 않습니다.
import os
import json
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict
# 자가평가는 고정 응답으로 노드의 입출력을 검사하며 추가 모델 호출을 하지 않습니다.
from types import SimpleNamespace
from unittest.mock import Mock, patch
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command, interrupt

material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY를 .env에 설정한 뒤 진행하세요.")
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)
print("사용 모델:", llm.model_name)

회의록 A·B와 작성 기준을 읽으세요. 담당자가 바뀐 일과 아직 정하지 않은 일을 구분합니다.

In [ ]:
# 원문 번호는 실행계획의 근거를 확인할 때 사용합니다.
meetings = json.loads((data_dir / "assignment01_meetings.json").read_text(encoding="utf-8"))
for meeting_id, minutes in meetings.items():
    print("회의:", meeting_id)
    print(minutes)
writing_rules = """확정 업무는 할 일, 담당자, 기한, 근거 문장 ID를 표로 정리한다.
담당자 변경은 최종 합의를 따른다. 미정인 담당자와 기한은 '확인 필요'로 남긴다.
합의한 업무의 미정 정보는 확인할 사항에도 쓴다. 업무의 선행 조건과 처리 순서도 보존한다. 제안·보류·취소 항목은 확정 업무와 구분한다.
미정 값을 정확히 표시했다면 문서 검토를 통과할 수 있다. 없는 정보를 채워 넣지 않는다."""
print("작성 기준:", writing_rules)

## 1. 실행계획과 검토 상태를 정의합니다

원문·최신 계획·검토 결과를 노드 사이에 전달합니다.

**작성:** 전체 코드 작성

**제공:** meetings, writing_rules와 공통 import를 제공합니다.

- `MeetingState(TypedDict)`에 minutes, rules, plan, feedback은 str, passed는 bool, review_count와 max_reviews는 int로 선언하세요.
- `meeting_input`에 A 회의 원문과 writing_rules를 넣고 plan·feedback은 빈 문자열, passed는 False, review_count는 0, max_reviews는 3으로 시작하세요.



<details><summary>힌트</summary>

```text
State는 필드 타입을, 초기 입력은 이번 실행의 실제 값을 담습니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** 원문과 작성 기준이 있고, 검토 횟수는 0입니다.

In [ ]:
# [출력 확인]
print("State 필드:", MeetingState.__annotations__)
print("초기 입력:", meeting_input)

**1번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 자가평가: State의 선언과 실제 초기값을 비교합니다.
from typing import get_type_hints, get_origin, get_args
grading_types = {"minutes": str, "rules": str, "plan": str, "feedback": str,
                 "passed": bool, "review_count": int, "max_reviews": int}
from typing_extensions import is_typeddict
assert is_typeddict(MeetingState), "MeetingState를 TypedDict로 선언하세요."
grading_annotations = get_type_hints(MeetingState)
assert all(grading_annotations.get(key) is value for key, value in grading_types.items()), "State 필드 이름과 타입을 확인하세요."
grading_initial = {"minutes": meetings["A"], "rules": writing_rules, "plan": "", "feedback": "",
                   "passed": False, "review_count": 0, "max_reviews": 3}
assert all(key in meeting_input and type(meeting_input[key]) is type(value) and meeting_input[key] == value
           for key, value in grading_initial.items()), "A 원문과 빈 계획·피드백, 초기 횟수와 상한을 확인하세요."
print("1번 통과: State와 초기 입력이 맞습니다.")
print("1번 자가평가 통과")

제공 프롬프트를 실행하세요. 작성 노드가 minutes와 rules를 전달할 때 사용합니다.

In [ ]:
# 제공: 합의된 업무와 아직 확정하지 않은 내용을 나누어 작성합니다.
plan_prompt = ChatPromptTemplate.from_messages([
    ("system", "회의록을 실행계획으로 정리하는 업무 비서입니다. 한국어로 짧게 작성하세요. "
     "표의 각 행에 원문 ID를 붙이고, 원문에 없는 담당자·기한·업무를 만들지 마세요.\n기준:\n{rules}"),
    ("human", "회의록:\n{minutes}"),
])

## 2. 실행계획 초안을 작성합니다

실행할 일과 확인할 일을 구분합니다.

**작성:** 코드 골격 완성

**제공:** plan_prompt와 함수 골격을 제공합니다.

- `plan_chain`에 plan_prompt와 llm을 연결하세요.
- `draft_plan`은 minutes·rules를 같은 이름의 프롬프트 입력으로 전달하세요.
- 모델 응답의 text를 plan 필드 하나로 반환하세요.

**확인:** 이 문항은 함수 정의입니다. 실제 초안은 7번에서 실행 후 확인합니다.

<details><summary>힌트</summary>

```text
State 전체 대신 체인에서 요구하는 두 입력을 전달합니다.
```

</details>

In [ ]:
# [작성] 체인 연결과 호출, 부분 갱신을 완성하세요.
plan_chain = ...
def draft_plan(state: MeetingState):
    """회의 원문에 근거한 실행계획을 반환합니다."""
    response = ...
    return ...

**2번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 자가평가 전용 고정 응답입니다. 실제 GPT 응답의 문장과 비교하지 않습니다.
from langchain_core.runnables import Runnable
assert isinstance(plan_chain, Runnable), "plan_prompt와 llm을 체인으로 연결하세요."
grading_prompt_input = {'minutes': '별도 원문', 'rules': '별도 기준'}
with patch.object(type(llm), "invoke", return_value=SimpleNamespace(text="연결 확인")) as grading_model:
    grading_response = plan_chain.invoke(grading_prompt_input)
    assert grading_model.call_count == 1, "제공 프롬프트와 모델을 연결하세요."
    grading_call = grading_model.call_args
    grading_prompt = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
    assert grading_prompt.to_messages() == plan_prompt.invoke(grading_prompt_input).to_messages(), "프롬프트에 받은 값을 전달하세요."
    assert grading_response.text == "연결 확인", "연결한 모델의 응답을 받으세요."
for grading_minutes, grading_rules, grading_text in [("회의 X의 원문", "X 작성 기준", "X 초안"), ("회의 Y의 원문", "Y 작성 기준", "Y 초안")]:
    grading_state = {**meeting_input, "minutes": grading_minutes, "rules": grading_rules, "plan": "기존 계획"}
    grading_before = dict(grading_state)
    grading_chain = Mock()
    grading_chain.invoke.return_value = SimpleNamespace(text=grading_text)
    with patch.dict(globals(), {"plan_chain": grading_chain, "llm": Mock(invoke=Mock(side_effect=AssertionError("자가평가에서는 GPT를 호출하지 않습니다.")))}):
        grading_update = draft_plan(grading_state)
        assert grading_chain.invoke.call_count == 1, "체인을 한 번 호출하세요."
        grading_call = grading_chain.invoke.call_args
        grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
        assert grading_payload == {"minutes": grading_minutes, "rules": grading_rules}, "지문에 지정한 입력 필드를 체인에 전달하세요."
        assert grading_update == {"plan": grading_text}, "응답의 text를 plan 하나로 반환하세요."
        assert grading_state == grading_before, "입력 State를 직접 변경하지 마세요."
print("2번 통과: 서로 다른 원문을 전달하고 계획만 반환합니다.")
print("2번 자가평가 통과")

제공 스키마와 검토 프롬프트를 실행하세요. 검토 결과는 passed와 feedback 두 필드로 받습니다.

In [ ]:
# 제공: 검토 결과의 형식과 근거 대조 기준입니다.
from pydantic import BaseModel, Field
class PlanReview(BaseModel):
    passed: bool = Field(description="원문과 기준을 지키면 true. 미정 정보를 정확히 표시한 것도 통과 가능")
    feedback: str = Field(description="틀린 내용과 고칠 방향. 문제없으면 통과 이유")

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "실행계획을 원문과 대조하세요. 최종 담당자, 기한, 확정 여부, 선행 조건과 처리 순서, 근거 ID, 누락 업무를 확인하세요. "
     "보류·취소를 실행 업무로 바꾸면 미통과입니다. 미정 정보를 확인 필요로 정확히 남겼으면 허용하세요. "
     "새 사실을 요구하지 마세요.\n기준:\n{rules}"),
    ("human", "회의록:\n{minutes}\n실행계획:\n{plan}"),
])

## 3. 실행계획을 검토하고 횟수를 갱신합니다

최신 계획의 사실과 구성을 검토합니다.

**작성:** 코드 골격 완성

**제공:** PlanReview와 review_prompt를 제공합니다.

- `review_chain`은 review_prompt에 PlanReview를 사용하는 구조화 출력 모델을 연결하세요. strict는 True입니다.
- `review_plan`은 minutes·rules·plan만 담은 딕셔너리를 체인에 전달하고 passed·feedback을 반환하세요.
- review_count는 입력보다 1 증가시켜 함께 반환합니다. 이 노드에서만 증가시킵니다.

**확인:** 검토할 때마다 횟수가 하나 증가하고 원문·계획은 그대로 유지됩니다. 실제 호출은 7번입니다.

<details><summary>힌트</summary>

```text
구조화 출력의 속성에서 판단과 의견을 꺼냅니다.
```

</details>

In [ ]:
# [작성] 구조화 출력 연결과 검토 결과 반환을 완성하세요.
review_chain = ...
def review_plan(state: MeetingState):
    """검토 판단과 의견, 완료한 검토 횟수를 반환합니다."""
    result = ...
    return ...

**3번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 구조화 검토의 통과·미통과 양쪽을 고정 응답으로 검사합니다.
assert isinstance(review_chain, Runnable) and review_chain.output_schema is PlanReview, "PlanReview 구조화 출력 체인을 연결하세요."
for grading_passed, grading_count in [(False, 0), (True, 2)]:
    grading_state = {**meeting_input, "minutes": f"원문 {grading_count}", "rules": "검토용 기준",
                     "plan": f"검토할 계획 {grading_count}", "review_count": grading_count}
    grading_before = dict(grading_state)
    grading_review = SimpleNamespace(passed=grading_passed, feedback=f"검토 의견 {grading_count}")
    grading_chain = Mock()
    grading_chain.invoke.return_value = grading_review
    with patch.dict(globals(), {"review_chain": grading_chain}):
        grading_update = review_plan(grading_state)
        assert grading_chain.invoke.call_count == 1, "체인을 한 번 호출하세요."
        grading_call = grading_chain.invoke.call_args
        grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
        assert grading_payload == {key: grading_before[key] for key in ("minutes", "rules", "plan")}, "지문에 지정한 입력 필드를 체인에 전달하세요."
        assert grading_update == {"passed": grading_passed, "feedback": grading_review.feedback, "review_count": grading_count + 1}, "모델 판단·의견과 1 증가한 검토 횟수만 반환하세요."
        assert type(grading_update["passed"]) is bool and type(grading_update["review_count"]) is int, "판단은 bool, 횟수는 int입니다."
        assert grading_state == grading_before, "검토 노드가 입력 State를 직접 변경하면 안 됩니다."
print("3번 통과: 판단과 의견을 보존하며 완료한 검토만 셉니다.")
print("3번 자가평가 통과")

제공 수정 프롬프트를 실행하세요. 원문·작성 기준·현재 계획·검토 의견을 입력받습니다.

In [ ]:
# 제공: 검토 의견을 반영하되 원문에 없는 사실은 보충하지 않습니다.
revision_prompt = ChatPromptTemplate.from_messages([
    ("system", "검토 의견에 따라 실행계획을 고치세요. 사실 근거는 회의록뿐입니다. "
     "확인 필요 항목을 임의로 확정하지 말고 근거 ID를 유지하세요.\n기준:\n{rules}"),
    ("human", "회의록:\n{minutes}\n현재 계획:\n{plan}\n검토 의견:\n{feedback}"),
])

## 4. 검토 의견으로 계획을 수정합니다

피드백이 다음 초안에 반영되어야 반복이 의미가 있습니다.

**작성:** 코드 골격 완성

**제공:** revision_prompt와 함수 골격을 제공합니다.

- `revision_chain`에 revision_prompt와 llm을 연결하세요.
- `revise_plan`은 minutes·rules·plan·feedback만 담은 딕셔너리를 체인에 전달하세요.
- 응답 text를 plan 하나로 반환하세요. review_count는 바꾸지 않습니다.

**확인:** 수정은 plan만 갱신하며 다음 검토 노드로 돌아갑니다.

<details><summary>힌트</summary>

```text
최신 초안과 검토 의견을 함께 전달해야 같은 실수를 고칠 수 있습니다.
```

</details>

In [ ]:
# [작성] 최신 계획과 의견을 읽는 수정 노드를 완성하세요.
revision_chain = ...
def revise_plan(state: MeetingState):
    """검토 의견을 반영한 최신 실행계획을 반환합니다."""
    response = ...
    return ...

**4번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 서로 다른 최신 계획과 피드백이 수정 체인에 전달되는지 검사합니다.
assert isinstance(revision_chain, Runnable), "revision_prompt와 llm을 연결하세요."
grading_prompt_input = {'minutes': '별도 원문', 'rules': '별도 기준', 'plan': '별도 계획', 'feedback': '별도 의견'}
with patch.object(type(llm), "invoke", return_value=SimpleNamespace(text="연결 확인")) as grading_model:
    grading_response = revision_chain.invoke(grading_prompt_input)
    assert grading_model.call_count == 1, "제공 프롬프트와 모델을 연결하세요."
    grading_call = grading_model.call_args
    grading_prompt = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
    assert grading_prompt.to_messages() == revision_prompt.invoke(grading_prompt_input).to_messages(), "프롬프트에 받은 값을 전달하세요."
    assert grading_response.text == "연결 확인", "연결한 모델의 응답을 받으세요."
for grading_number in (1, 2):
    grading_state = {**meeting_input, "minutes": f"회의 {grading_number}", "rules": f"기준 {grading_number}",
                     "plan": f"최신 계획 {grading_number}", "feedback": f"고칠 내용 {grading_number}", "review_count": grading_number}
    grading_before = dict(grading_state)
    grading_chain = Mock()
    grading_chain.invoke.return_value = SimpleNamespace(text=f"수정한 계획 {grading_number}")
    with patch.dict(globals(), {"revision_chain": grading_chain}):
        grading_update = revise_plan(grading_state)
        assert grading_chain.invoke.call_count == 1, "체인을 한 번 호출하세요."
        grading_call = grading_chain.invoke.call_args
        grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
        assert grading_payload == {key: grading_before[key] for key in ("minutes", "rules", "plan", "feedback")}, "지문에 지정한 입력 필드를 체인에 전달하세요."
        assert grading_update == {"plan": f"수정한 계획 {grading_number}"}, "수정한 plan만 반환하고 검토 횟수는 바꾸지 마세요."
        assert grading_state == grading_before, "수정 노드는 입력 State를 직접 변경하지 않습니다."
print("4번 통과: 최신 계획과 의견을 전달하고 계획만 갱신합니다.")
print("4번 자가평가 통과")

## 5. 반복과 종료를 결정합니다

미통과 문서도 상한에 도달하면 사람이 확인해야 합니다.

**작성:** 전체 코드 작성

**제공:** MeetingState와 경계 조건 확인 셀을 제공합니다.

- `choose_plan_route(state: MeetingState)` 전체를 작성하세요.
- passed가 True이거나 review_count가 max_reviews 이상이면 `end`, 나머지는 `revise`를 반환하세요.
- 모델을 호출하지 않고 State도 변경하지 않습니다.



<details><summary>힌트</summary>

```text
통과 여부와 상한 도달 여부는 각각 종료 이유가 될 수 있습니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** 아래 세 경우는 차례로 end, revise, end입니다. max_reviews는 1 이상의 정수로 사용합니다.

In [ ]:
# [출력 확인]
for passed, count, limit in [(True, 1, 3), (False, 1, 3), (False, 1, 1)]:
    state = {**meeting_input, "passed": passed, "review_count": count, "max_reviews": limit}
    print(passed, count, limit, "->", choose_plan_route(state))

**5번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 상한 미만·도달·초과와 조기 통과를 함께 확인합니다. 모델 호출은 없습니다.
for grading_passed, grading_count, grading_limit, grading_expected in [
    (True, 1, 3, "end"), (False, 1, 3, "revise"),
    (False, 3, 3, "end"), (False, 4, 3, "end"),
    (False, 1, 1, "end"), (False, 2, 5, "revise")]:
    grading_state = {**meeting_input, "passed": grading_passed, "review_count": grading_count, "max_reviews": grading_limit}
    grading_before = dict(grading_state)
    with patch.dict(globals(), {name: Mock(invoke=Mock(side_effect=AssertionError("경로 선택에서 모델을 호출하면 안 됩니다.")))
                                for name in ("llm", "plan_chain", "review_chain", "revision_chain")}):
        grading_route = choose_plan_route(grading_state)
        assert grading_route == grading_expected, f"passed={grading_passed}, count={grading_count}, limit={grading_limit}의 경로를 확인하세요."
        assert grading_state == grading_before, "경로 선택은 State를 읽기만 합니다."
print("5번 통과: 두 종료 조건과 남은 수정 경로가 맞습니다.")
print("5번 자가평가 통과")

## 6. 검토 사이클과 체크포인터를 연결합니다

수정 과정을 저장해 초안과 최종본을 비교합니다.

**작성:** 코드 골격 완성

**제공:** 노드 등록과 별도 변수 이름을 제공합니다.

- `meeting_builder`에 MeetingState를 사용하세요.
- START → draft_plan → review_plan을 연결합니다.
- review_plan 뒤에 choose_plan_route를 연결하고 revise는 revise_plan, end는 END에 대응하세요.
- revise_plan에서 review_plan으로 돌아갑니다.
- `meeting_saver`는 InMemorySaver 객체입니다. 이를 체크포인터로 연결해 `meeting_graph`로 컴파일하세요.

**확인:** 한 수정 사이클과 두 종료 조건을 갖습니다. 컴파일은 모델을 호출하지 않습니다.

<details><summary>힌트</summary>

```text
조건부 분기가 있는 review_plan 뒤에 별도의 고정 엣지를 추가하지 않습니다.
```

</details>

In [ ]:
# [작성] 반복 경로와 상태 저장을 연결하세요.
meeting_builder = ...
meeting_builder.add_node("draft_plan", draft_plan)
meeting_builder.add_node("review_plan", review_plan)
meeting_builder.add_node("revise_plan", revise_plan)
...
meeting_saver = ...
meeting_graph = ...

**6번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 고정 응답으로 실제 그래프 연결을 검사합니다. 이 검사는 GPT를 호출하지 않습니다.
from uuid import uuid4 as grading_uuid4
grading_expected_edges = {(START, "draft_plan"), ("draft_plan", "review_plan"),
                          ("review_plan", "revise_plan"), ("review_plan", END), ("revise_plan", "review_plan")}
grading_edges = {(edge.source, edge.target) for edge in meeting_graph.get_graph().edges}
assert grading_edges == grading_expected_edges, "작성→검토, 조건부 수정/종료, 수정→검토 연결을 확인하세요."
assert isinstance(meeting_saver, InMemorySaver) and meeting_graph.checkpointer is meeting_saver, "meeting_saver를 checkpointer로 연결하세요."
for grading_verdicts, grading_limit, grading_revisions in [([False, True], 3, 1), ([False, False, False], 3, 2), ([False], 1, 0)]:
    grading_draft = Mock()
    grading_draft.invoke.return_value = SimpleNamespace(text="자가평가용 초안")
    grading_review = Mock()
    grading_review.invoke.side_effect = [SimpleNamespace(passed=value, feedback=f"의견 {number}") for number, value in enumerate(grading_verdicts, 1)]
    grading_revision = Mock()
    grading_revision.invoke.side_effect = [SimpleNamespace(text=f"자가평가용 수정 {number}") for number in range(1, grading_revisions + 1)]
    grading_config = {"configurable": {"thread_id": f"grading-meeting-{grading_uuid4()}"}}
    grading_input = {**meeting_input, "max_reviews": grading_limit}
    grading_before = dict(grading_input)
    with patch.dict(globals(), {"plan_chain": grading_draft, "review_chain": grading_review, "revision_chain": grading_revision}):
        grading_result = meeting_graph.invoke(grading_input, grading_config)
        assert grading_draft.invoke.call_count == 1, "초안은 한 번만 작성해야 합니다."
        assert grading_review.invoke.call_count == len(grading_verdicts) and grading_revision.invoke.call_count == grading_revisions, "통과 또는 상한까지 수정과 재검토가 연결되어야 합니다."
        assert grading_result["review_count"] == len(grading_verdicts) and grading_result["passed"] is grading_verdicts[-1], "최종 검토 횟수와 판단을 보존하세요."
        assert grading_result["plan"] == (f"자가평가용 수정 {grading_revisions}" if grading_revisions else "자가평가용 초안"), "가장 최근의 수정 계획이 최종 State에 남아야 합니다."
        for grading_index in range(grading_revisions):
            grading_call = grading_revision.invoke.call_args_list[grading_index]
            grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
            assert grading_payload["feedback"] == f"의견 {grading_index + 1}", "각 검토 의견이 다음 수정으로 전달되어야 합니다."
        assert grading_input == grading_before, "그래프 입력값을 직접 변경하면 안 됩니다."
        assert meeting_graph.get_state(grading_config).next == (), "종료 후 다음 노드는 없어야 합니다."
print("6번 통과: 수정 후 통과, 상한 종료, 1회 검토가 모두 연결됩니다. (GPT 추가 호출 0회)")
print("6번 자가평가 통과")

## 7. A 회의를 실행하고 종료 이유를 읽습니다

계획의 문장보다 원문 근거를 먼저 확인합니다.

**작성:** 코드 골격 완성

**제공:** 독립 작업 ID와 결과 출력 코드를 제공합니다.

- `meeting_result`에 meeting_graph를 meeting_input과 meeting_config로 실행한 결과를 담으세요.
- `plan_status`는 passed가 참이면 `문서 기준 충족`, 아니면 `검토 상한 도달: 사람 확인 필요`입니다.
- A4의 담당자 변경, A5의 미정 항목과 데이터 정제 후 날짜를 정한다는 순서, A6의 미확정 제안, A7의 개인정보 점검을 원문과 대조하세요.



<details><summary>힌트</summary>

```text
invoke의 결과에서 종료 판단을 읽습니다. 최초 입력의 passed는 그대로 False입니다.
```

</details>

In [ ]:
# [작성] 실행 결과와 종료 이유를 구하세요.
from uuid import uuid4
meeting_config = {"configurable": {"thread_id": f"meeting-A-{uuid4()}"}}
meeting_result = ...
plan_status = ...

**결과 확인:** 검토 횟수는 1~3입니다. 데이터 정제 담당자는 서연이고 자동 답변 기능은 확정 업무가 아닙니다.

In [ ]:
# [출력 확인]
print("종료 이유:", plan_status)
print("검토 횟수:", meeting_result["review_count"])
print("검토 의견:", meeting_result["feedback"])
display(Markdown(meeting_result["plan"]))
(output_dir / "assignment01_plan_A.md").write_text(
    meeting_result["plan"] + "\n\n" + plan_status + "\n" + meeting_result["feedback"], encoding="utf-8",
)
print("초기 입력의 검토 횟수:", meeting_input["review_count"])

**7번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 이미 실행한 A 결과를 검사합니다. 검토 통과 여부는 정답으로 강제하지 않습니다.
assert meeting_result["minutes"] == meetings["A"] and meeting_result["rules"] == writing_rules, "A의 원문과 기준을 전달하세요."
assert isinstance(meeting_result["plan"], str) and meeting_result["plan"].strip(), "작성된 계획이 필요합니다."
assert type(meeting_result["passed"]) is bool and isinstance(meeting_result["feedback"], str), "검토 결과의 타입을 확인하세요."
assert type(meeting_result["review_count"]) is int and 1 <= meeting_result["review_count"] <= 3, "검토 횟수는 1~3회입니다."
assert meeting_result["passed"] or meeting_result["review_count"] == 3, "미통과라면 검토 상한에 도달한 뒤 종료해야 합니다."
assert plan_status == ("문서 기준 충족" if meeting_result["passed"] else "검토 상한 도달: 사람 확인 필요"), "종료 이유는 최종 passed로 구하세요."
assert meeting_graph.get_state(meeting_config).values == meeting_result, "meeting_config의 저장 상태와 실행 결과가 일치해야 합니다."
assert meeting_input["plan"] == "" and meeting_input["review_count"] == 0 and meeting_input["passed"] is False, "최초 입력을 결과로 덮어쓰지 마세요."
print("7번 통과: A 실행 결과와 종료 이유가 일치합니다. 내용은 원문과 직접 대조하세요.")
print("7번 자가평가 통과")

## 8. 다른 회의를 한 번만 검토합니다

서로 다른 작업이 섞이지 않는지와 최소 상한을 함께 확인합니다.

**작성:** 코드 골격 완성

**제공:** B 원문과 초기 입력 생성 골격을 제공합니다.

- `meeting_b_input`은 meeting_input을 바탕으로 minutes만 B 원문으로, max_reviews만 1로 바꾸세요. meeting_result를 복사하지 않습니다.
- 새 thread_id의 `meeting_b_config`로 실행해 `meeting_b_result`에 담으세요.
- A 상태를 같은 meeting_config로 조회해 `saved_a`에 담으세요.



<details><summary>힌트</summary>

```text
결과가 채워진 State 대신 빈 초기 State에서 별도 작업을 시작합니다.
```

</details>

In [ ]:
# [작성] B의 독립 실행과 A 조회를 완성하세요.
meeting_b_input = ...
meeting_b_config = {"configurable": {"thread_id": f"meeting-B-{uuid4()}"}}
meeting_b_result = ...
saved_a = ...

**결과 확인:** B review_count는 1이고 A 계획은 이전 결과와 같습니다. B에는 계정 20개·강의실 예약·사전 안내가 있고 외부 강사 섭외는 제외됩니다.

In [ ]:
# [출력 확인]
print("B 검토 횟수:", meeting_b_result["review_count"])
display(Markdown(meeting_b_result["plan"]))
print("A 계획 유지:", saved_a.values["plan"] == meeting_result["plan"])

**8번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# A·B의 작업 ID와 기존 실행 결과를 비교합니다. 모델을 다시 호출하지 않습니다.
grading_expected_b = {**meeting_input, "minutes": meetings["B"], "max_reviews": 1}
assert meeting_b_input == grading_expected_b, "빈 초기 입력에서 원문과 상한만 바꾸세요."
assert meeting_b_config["configurable"]["thread_id"] != meeting_config["configurable"]["thread_id"], "A와 B에는 서로 다른 thread_id를 사용하세요."
assert meeting_b_result["minutes"] == meetings["B"] and meeting_b_result["review_count"] == 1, "B의 원문으로 한 번만 검토해야 합니다."
assert isinstance(meeting_b_result["plan"], str) and meeting_b_result["plan"].strip(), "B의 실행계획이 필요합니다."
assert meeting_graph.get_state(meeting_b_config).values == meeting_b_result, "B의 설정으로 실행 결과를 저장하세요."
assert saved_a.values == meeting_result and saved_a.config["configurable"]["thread_id"] == meeting_config["configurable"]["thread_id"], "saved_a는 A의 get_state 결과여야 합니다."
print("8번 통과: B는 한 번 검토하고 A의 기록은 유지합니다.")
print("8번 자가평가 통과")

## 9. 초안부터 최종 상태까지 이력을 읽습니다

어떤 피드백이 어떤 수정으로 이어졌는지 확인합니다.

**작성:** 코드 골격 완성

**제공:** 이력 출력과 문서 비교 코드를 제공합니다.

- `current`는 A의 get_state 결과, `history`는 같은 A의 get_state_history 결과를 list로 바꾼 값입니다.
- get_state_history는 최신순입니다. 제공 코드가 시간순으로 뒤집어 계획이 있는 기록을 읽습니다.
- values·next·config를 확인하고, 첫 초안과 최종 계획을 비교하세요. 조회는 모델을 호출하지 않습니다.



<details><summary>힌트</summary>

```text
전체 스냅샷의 속성과 그 안의 State 필드를 구분합니다.
```

</details>

In [ ]:
# [작성] 현재 상태와 최신순 이력을 조회하세요.
current = ...
history = ...

**결과 확인:** 완료 상태의 next는 빈 튜플입니다. 이력 수는 검토 횟수와 같을 필요가 없고, 첫 검토에 통과하면 초안과 최종본이 같을 수 있습니다.

In [ ]:
# [출력 확인]
print("다음 노드:", current.next)
print("체크포인트 설정:", current.config)
plan_history = [snapshot for snapshot in reversed(history) if snapshot.values.get("plan")]
for snapshot in plan_history:
    print("검토 횟수:", snapshot.values["review_count"], "/ 다음:", snapshot.next)
    print("검토 의견:", snapshot.values["feedback"])
display(Markdown("### 첫 초안\n" + plan_history[0].values["plan"]))
display(Markdown("### 최종 계획\n" + current.values["plan"]))

**9번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 이력의 개수를 고정하지 않고 실제 저장된 A의 이력과 대조합니다.
grading_saved = meeting_graph.get_state(meeting_config)
grading_history = list(meeting_graph.get_state_history(meeting_config))
assert current.values == grading_saved.values and current.config == grading_saved.config and current.next == (), "current는 완료된 A의 현재 스냅샷이어야 합니다."
assert isinstance(history, list) and len(history) == len(grading_history), "A의 전체 이력을 list로 읽으세요."
assert all(actual.config == expected.config and actual.values == expected.values and actual.next == expected.next
           for actual, expected in zip(history, grading_history)), "history는 같은 A의 get_state_history가 반환한 최신순 이력입니다."
grading_plans = [saved for saved in reversed(history) if saved.values.get("plan")]
assert grading_plans and grading_plans[0].values["review_count"] == 0, "검토 이전의 첫 초안도 이력에 있어야 합니다."
assert grading_plans[-1].values["plan"] == meeting_result["plan"], "마지막 계획은 A의 최종 결과와 같아야 합니다."
print("9번 통과: 현재 상태와 최신순 이력을 읽었습니다.")
print("9번 자가평가 통과")

### 제공 실습: 수정과 재검토를 직접 확인합니다

A가 첫 검토에서 통과하면 수정 노드를 실행하지 않았을 수 있습니다. 이번에는 일부러 담당자와 기한을 잘못 적은 초안과 구체적인 검토 의견을 제공합니다. 한 차례 검토를 받은 상태를 가정해 review_count는 1로 시작합니다. **수정 → 재검토** 노드를 직접 호출해 변경된 계획을 다음 노드에 전달합니다. 그대로 실행하세요. GPT를 2회 호출합니다.

앞서 저장한 A의 결과와 이력은 유지됩니다. 수정 전후를 읽고 담당자가 서연, 기한이 10월 8일로 바뀌었는지 확인하세요. 재검토가 미통과라면 남은 의견을 읽습니다. 이 실습은 모델의 통과 여부를 정답으로 강제하지 않습니다.

In [ ]:
# 제공: 실제 모델로 수정 노드와 재검토 노드를 각각 한 번 실행합니다.
revision_demo_input = {
    **meeting_input,
    "plan": "| 할 일 | 담당자 | 기한 | 근거 |\n|---|---|---|---|\n| 테스트 데이터 정제 | 민호 | 10월 9일 | A3 |",
    "feedback": "A4의 최종 합의에 따라 정제 담당자는 서연, 기한은 10월 8일입니다. "
                "이를 고치고 A2·A5·A7의 합의된 업무와 확인할 사항도 복원하세요. A6은 미확정 제안입니다.",
    "review_count": 1,
}
revision_demo_update = revise_plan(revision_demo_input)
revision_demo_state = {**revision_demo_input, **revision_demo_update}
revision_demo_review = review_plan(revision_demo_state)
display(Markdown("### 의도적으로 오류를 넣은 초안\n" + revision_demo_input["plan"]))
display(Markdown("### 수정한 계획\n" + revision_demo_state["plan"]))
print("재검토 결과:", revision_demo_review["passed"])
print("남은 검토 의견:", revision_demo_review["feedback"])
print("완료한 검토 횟수:", revision_demo_review["review_count"])

## 10. 회의 후속 상담 State를 작성합니다

비서가 앞 질문을 기억하면서 회의 내용에 답하도록 합니다.

**작성:** 전체 코드 작성

**제공:** Annotated·AnyMessage·add_messages를 공통 준비에서 제공합니다.

- `MeetingChatState(TypedDict)` 전체를 선언하세요.
- minutes와 plan은 str, messages는 AnyMessage 리스트에 add_messages를 적용한 누적 필드입니다.



<details><summary>힌트</summary>

```text
일반 문자열 필드는 최신 값으로 교체되며 messages에는 합치는 규칙을 붙입니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** 원문·계획은 유지하고 새 질문과 답변은 messages에 누적할 수 있습니다.

In [ ]:
# [출력 확인]
print("상담 State:", MeetingChatState.__annotations__)

**10번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# messages에 누적 규칙이 붙었는지 타입 선언으로 확인합니다.
from typing import get_type_hints, get_origin, get_args
from typing_extensions import is_typeddict
assert is_typeddict(MeetingChatState), "MeetingChatState를 TypedDict로 선언하세요."
grading_annotations = get_type_hints(MeetingChatState, include_extras=True)
assert grading_annotations.get("minutes") is str and grading_annotations.get("plan") is str, "minutes와 plan은 str입니다."
grading_messages = grading_annotations.get("messages")
assert get_origin(grading_messages) is Annotated, "messages에 Annotated로 add_messages를 지정하세요."
grading_arguments = get_args(grading_messages)
assert get_origin(grading_arguments[0]) is list and get_args(grading_arguments[0]) == (AnyMessage,), "messages는 AnyMessage 리스트입니다."
assert add_messages in grading_arguments[1:], "messages의 누적 규칙은 add_messages입니다."
print("10번 통과: 원문·계획과 누적 대화 필드가 준비되었습니다.")
print("10번 자가평가 통과")

상담 프롬프트를 준비하세요. 원문·계획과 누적 대화를 함께 읽습니다.

In [ ]:
# 제공: 원문을 우선하며 계획에 남은 확인 사항을 설명합니다.
meeting_chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "회의 후속 업무 비서입니다. 마지막 질문에 짧게 답하세요. "
     "원문과 계획이 다르면 원문을 따르세요. 미정 정보를 확정하지 말고 자료에 있는 근거 ID를 쓰세요.\n"
     "원문:\n{minutes}\n계획:\n{plan}"),
    MessagesPlaceholder("messages"),
])

## 11. 대화 기록을 읽는 상담 그래프를 만듭니다

매번 원문을 보내지 않아도 같은 상담을 이어갑니다.

**작성:** 코드 골격 완성

**제공:** 프롬프트·노드 골격·노드 등록을 제공합니다.

- `meeting_chat_chain`에 프롬프트와 llm을 연결하세요.
- `answer_meeting`은 minutes·plan·messages를 체인에 전달하고 새 응답 한 개만 messages 리스트로 반환하세요.
- `chat_builder`는 MeetingChatState를 사용하며 START → answer_meeting → END입니다.
- `chat_saver`는 새 InMemorySaver이며, 이를 연결한 `chat_graph`를 만드세요.

**확인:** 이 문항에서는 모델을 호출하지 않습니다. 기존 메시지를 반환값에 다시 붙이지 않습니다.

<details><summary>힌트</summary>

```text
기존 대화는 State에서 읽고 반환할 때는 이번 응답만 보냅니다.
```

</details>

In [ ]:
# [작성] 상담 노드와 체크포인터 연결을 완성하세요.
meeting_chat_chain = ...
def answer_meeting(state: MeetingChatState):
    """회의 근거와 이전 대화를 읽어 새 답변을 반환합니다."""
    reply = ...
    return ...

chat_builder = ...
chat_builder.add_node("answer_meeting", answer_meeting)
...
chat_saver = ...
chat_graph = ...

**11번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 고정 AIMessage를 사용해 상담 노드와 그래프를 검사합니다. GPT 호출은 없습니다.
from langchain_core.messages import AIMessage
from copy import deepcopy
grading_prompt_input = {'minutes': '별도 원문', 'plan': '별도 계획', 'messages': []}
with patch.object(type(llm), "invoke", return_value=SimpleNamespace(text="연결 확인")) as grading_model:
    grading_response = meeting_chat_chain.invoke(grading_prompt_input)
    assert grading_model.call_count == 1, "제공 프롬프트와 모델을 연결하세요."
    grading_call = grading_model.call_args
    grading_prompt = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
    assert grading_prompt.to_messages() == meeting_chat_prompt.invoke(grading_prompt_input).to_messages(), "프롬프트에 받은 값을 전달하세요."
    assert grading_response.text == "연결 확인", "연결한 모델의 응답을 받으세요."
assert {(edge.source, edge.target) for edge in chat_graph.get_graph().edges} == {(START, "answer_meeting"), ("answer_meeting", END)}, "상담 그래프는 START→answer_meeting→END입니다."
assert isinstance(chat_saver, InMemorySaver) and chat_graph.checkpointer is chat_saver and chat_saver is not meeting_saver, "상담용 새 InMemorySaver를 연결하세요."
for grading_number in (1, 2):
    grading_state = {"minutes": f"회의 {grading_number}", "plan": f"계획 {grading_number}",
                     "messages": [HumanMessage(content=f"질문 {grading_number}")]}
    grading_before = deepcopy(grading_state)
    grading_reply = AIMessage(content=f"고정 답변 {grading_number}")
    grading_chain = Mock()
    grading_chain.invoke.return_value = grading_reply
    with patch.dict(globals(), {"meeting_chat_chain": grading_chain}):
        grading_update = answer_meeting(grading_state)
        assert grading_chain.invoke.call_count == 1, "체인을 한 번 호출하세요."
        grading_call = grading_chain.invoke.call_args
        grading_payload = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
        assert grading_payload == grading_before, "지문에 지정한 입력 필드를 체인에 전달하세요."
        assert grading_update == {"messages": [grading_reply]}, "새 AI 응답 한 개만 messages 리스트로 반환하세요."
        assert grading_state == grading_before, "기존 State와 messages를 직접 변경하지 마세요."
print("11번 통과: 상담 노드가 근거와 대화를 읽고 새 답변만 반환합니다.")
print("11번 자가평가 통과")

## 12. 같은 상담을 이어가고 다른 회의와 분리합니다

대명사로 물어도 앞 대화의 업무를 이어 설명해야 합니다.

**작성:** 코드 골격 완성

**제공:** 두 상담의 ID·첫 입력·후속 질문을 제공합니다.

- A 첫 입력을 chat_a_config로 실행해 chat_a_first에 담으세요.
- 같은 설정에 제공된 followup_input만 전달해 chat_a_followup에 담으세요. 원문이나 이전 메시지를 다시 넣지 않습니다.
- B 입력은 chat_b_config로 실행해 chat_b_result에 담으세요.
- A는 데이터 정제 담당자·기한, B는 사전 안내의 미정 항목을 원문과 대조하세요.



<details><summary>힌트</summary>

```text
같은 체크포인터에서 thread_id를 유지하면 후속 입력에 새 메시지만 보낼 수 있습니다.
```

</details>

In [ ]:
# [작성] 첫 질문, 같은 상담의 후속 질문, 별도 상담을 실행하세요.
chat_a_config = {"configurable": {"thread_id": f"meeting-chat-A-{uuid4()}"}}
chat_b_config = {"configurable": {"thread_id": f"meeting-chat-B-{uuid4()}"}}
chat_a_input = {"minutes": meetings["A"], "plan": meeting_result["plan"], "messages": [HumanMessage(content="데이터 정제는 누가 맡기로 했나요?")]}
followup_input = {"messages": [HumanMessage(content="그 작업의 기한도 알려 주세요.")]}
chat_b_input = {"minutes": meetings["B"], "plan": meeting_b_result["plan"], "messages": [HumanMessage(content="사전 안내에서 아직 정하지 않은 것은 무엇인가요?")]}
chat_a_first = ...
chat_a_followup = ...
chat_b_result = ...

**결과 확인:** 12번 셀을 처음 실행하면 A 첫 답변은 메시지 2개, A 후속 답변은 4개, B 답변은 2개입니다. A의 '그 작업'은 데이터 정제이며 담당자는 서연, 기한은 10월 8일입니다.

In [ ]:
# [출력 확인]
for label, result in [("A 첫 답변", chat_a_first), ("A 후속 답변", chat_a_followup), ("B 답변", chat_b_result)]:
    print(label, "/ 메시지 수:", len(result["messages"]))
    display(Markdown(result["messages"][-1].text))

**12번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 이전 실행의 메시지 역할·연속성과 자료 분리를 확인합니다. 답변 문장은 채점하지 않습니다.
assert chat_a_config["configurable"]["thread_id"] != chat_b_config["configurable"]["thread_id"], "서로 다른 회의는 별도 thread_id를 사용하세요."
for grading_result, grading_input, grading_roles in [
    (chat_a_first, chat_a_input, ["human", "ai"]),
    (chat_a_followup, chat_a_input, ["human", "ai", "human", "ai"]),
    (chat_b_result, chat_b_input, ["human", "ai"])]:
    assert grading_result["minutes"] == grading_input["minutes"] and grading_result["plan"] == grading_input["plan"], "각 상담의 원문과 계획을 보존하세요."
    assert [message.type for message in grading_result["messages"]] == grading_roles, "질문과 답변을 누적하세요. A는 2→4개, B는 2개입니다."
    assert all(message.text.strip() for message in grading_result["messages"]), "빈 질문이나 답변이 없어야 합니다."
assert chat_a_followup["messages"][:2] == chat_a_first["messages"], "A 후속 질문은 앞 질문과 답변을 이어야 합니다."
assert chat_a_first["messages"][0].text == chat_a_input["messages"][0].text, "A의 첫 입력에 제공된 질문을 보내세요."
assert chat_a_followup["messages"][2].text == followup_input["messages"][0].text, "후속 질문만 새 메시지로 보내세요."
assert chat_b_result["messages"][0].text == chat_b_input["messages"][0].text, "B의 첫 질문을 B 상담에 보내세요."
assert chat_graph.get_state(chat_a_config).values == chat_a_followup and chat_graph.get_state(chat_b_config).values == chat_b_result, "두 설정에 각 대화의 최종 상태가 저장되어야 합니다."
print("12번 통과: A의 대화는 이어지고 B의 대화와 분리됩니다. 답변의 사실은 원문과 대조하세요.")
print("12번 자가평가 통과")

## 마무리합니다

- 원문과 최종 계획을 대조하고 자동 검토가 놓친 내용이 있는지 적어 보세요.
- 검토 상한 종료와 문서 기준 충족을 구분해 설명하세요.
- A·B 회의의 상태와 상담 대화가 섞이지 않았는지 확인하세요.
- 제출: 완성한 노트북, `assignment01_plan_A.md`, 초안과 최종본의 차이 및 종료 이유를 적은 짧은 메모.